In [ ]:
# === IMPORTS : on charge les bibliothèques nécessaires ===
import numpy as np                # np : calcul numérique (tableaux, maths)
import pandas as pd               # pd : tableaux de données (le "df")
import matplotlib.pyplot as plt   # plt : graphiques
from scipy import stats           # stats : tests statistiques (t-test, normalité)

In [ ]:
# === 1. CHARGEMENT DES DONNÉES ===
# pd.read_csv : lit le fichier CSV et le transforme en DataFrame (un tableau)
# parse_dates=["Date"] : la colonne Date est lue comme une VRAIE date (pas du texte)
# dayfirst=True : format JJ/MM/AAAA (européen), sinon pandas lit à l'américaine
df = pd.read_csv("data.csv", parse_dates=["Date"], dayfirst=True)

# df.head() : affiche les 5 premières lignes pour vérifier
df.head()

In [ ]:
# === MISE EN PLACE DE L'INDEX TEMPOREL ===
# set_index : la colonne Date devient l'étiquette de chaque ligne
df = df.set_index("Date")
# sort_index : on trie du plus ancien au plus récent
df = df.sort_index()

# df.index.min()/max() : date la plus ancienne / la plus récente
# .date() : garde juste la date (sans l'heure)
print("Period covered:", df.index.min().date(), " = ", df.index.max().date())
# len(df) : nombre de lignes (jours de bourse)
print("Number of records:", len(df))
df.head()

In [ ]:
# === 2. VISUALISATION : courbe du prix de clôture ===
# figure(figsize=(14,6)) : crée une figure large (14) et basse (6)
plt.figure(figsize=(14, 6))
# plot(x, y) : trace une courbe -> x = les dates, y = la colonne Close
plt.plot(df.index, df["Close"], label="Closing Price", color="blue")
plt.title("Apple(AAPL) Closing (1981-2023)")  # titre
plt.xlabel("Year")                 # étiquette axe X
plt.ylabel("Closing price (USD)")  # étiquette axe Y
plt.grid(True)                     # affiche la grille de fond
plt.tight_layout()                 # ajuste les marges pour ne rien couper
plt.show()                         # affiche le graphique

In [ ]:
# === Graphique en chandeliers : bibliothèque spécialisée ===
import mplfinance as mpf  # mpf : graphiques boursiers (chandeliers japonais)

In [ ]:
# tail(60) : on prend les 60 derniers jours (sinon 10 608 chandelles = illisible)
recent = df.tail(60)
# mpf.plot : trace le graphique
#   type="candle"   : chandelier (chaque bougie = Open/High/Low/Close du jour)
#   style="charles" : thème de couleurs prédéfini
mpf.plot(recent, type="candle", style="charles",
         title="Apple(AAPL) Last 60 Days", ylabel="Price (USD)")

In [ ]:
# === 3. STATISTIQUES DESCRIPTIVES ===
# df[[...]] : sélectionne plusieurs colonnes (double crochets = liste de colonnes)
# .agg([...]) : applique plusieurs calculs d'un coup
#   mean = moyenne, median = valeur du milieu, std = écart-type (dispersion)
df[["Open", "High", "Low", "Close", "Volume"]].agg(["mean", "median", "std"])

In [ ]:
# === MOYENNES MOBILES (rolling) ===
# rolling(window=50) : fenêtre glissante de 50 jours
# .mean() : moyenne de chaque fenêtre -> courbe lissée (MA50 = court terme)
df["MA50"] = df["Close"].rolling(window=50).mean()
# MA200 : tendance de fond (long terme)
df["MA200"] = df["Close"].rolling(window=200).mean()

# On trace le prix + les 2 moyennes mobiles
plt.figure(figsize=(14, 6))
# alpha=0.5 : courbe semi-transparente
plt.plot(df.index, df["Close"], label="Close", alpha=0.5)
plt.plot(df.index, df["MA50"], label="50-day MA")
plt.plot(df.index, df["MA200"], label="200-day MA")
plt.title("AAPL - Closing Price with Moving Averages")
plt.xlabel("Year")
plt.ylabel("Price (USD)")
plt.legend()        # affiche la légende (les "label")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# === 4. TEST D'HYPOTHÈSE : t-test ===
# df.loc["2020", "Close"] : sélectionne le Close de TOUTE l'année 2020
close_2020 = df.loc["2020", "Close"]
close_2021 = df.loc["2021", "Close"]

# ttest_ind : t-test à 2 échantillons indépendants (compare 2 moyennes)
# renvoie 2 valeurs : la statistique t et la p-value
t_stat, p_value = stats.ttest_ind(close_2020, close_2021)

print("t-statistic:", t_stat)
print("p-value:", p_value)

# H0 = "les 2 moyennes sont égales"
# Si p-value < 0.05 -> différence significative (on rejette H0)
alpha = 0.05
if p_value < alpha:
    print("Significant difference between 2020 and 2021 (reject H0).")
else:
    print("No significant difference (fail to reject H0).")

In [ ]:
# === RENDEMENTS QUOTIDIENS + TEST DE NORMALITÉ ===
# pct_change() : variation en % d'un jour à l'autre = (jour - veille) / veille
df["Daily_Return"] = df["Close"].pct_change()

# Histogramme des rendements
plt.figure(figsize=(12, 6))
# dropna() : enlève les valeurs manquantes (le 1er jour n'a pas de "veille")
# bins=100 : 100 barres ; edgecolor : contour noir
plt.hist(df["Daily_Return"].dropna(), bins=100, edgecolor="black")
plt.title("AAPL - Distribution of Daily Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.grid(True)
plt.tight_layout()
plt.show()

# normaltest (D'Agostino-Pearson) : les rendements suivent-ils une loi normale ?
# (adapté aux grands échantillons, contrairement à Shapiro)
returns = df["Daily_Return"].dropna()
stat, p_value = stats.normaltest(returns)

print("Normality test statistic:", stat)
print("p-value:", p_value)

# Si p-value < 0.05 -> PAS normal (fat tails : krachs plus fréquents que prévu)
alpha = 0.05
if p_value < alpha:
    print("Daily returns are NOT normally distributed (reject H0).")
else:
    print("Daily returns look normally distributed (fail to reject H0).")

In [ ]:
# === 5. BONUS : moyenne mobile "à la main" par convolution (NumPy) ===
window = 50
# np.ones(50) : tableau de 50 fois "1" ; / window -> [0.02, ...] = noyau de moyenne
kernel = np.ones(window) / window

# .values : convertit la colonne pandas en tableau NumPy brut
close_values = df["Close"].values
# np.convolve : glisse le noyau sur les prix = moyenne mobile (traitement du signal)
# mode="valid" : ne garde que les positions entièrement recouvertes (pas les bords)
ma_convolve = np.convolve(close_values, kernel, mode="valid")

# On aligne les dates : la convolution est plus courte de 49 points
ma_dates = df.index[window - 1:]

plt.figure(figsize=(14, 6))
plt.plot(df.index, close_values, label="Close", alpha=0.5)
plt.plot(ma_dates, ma_convolve, label="50-day MA (convolution)", color="red")
plt.title("AAPL - Moving Average via Convolution")
plt.xlabel("Year")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# === CORRÉLATION MA50 vs Volume (NumPy) ===
# dropna() : on enlève les premières lignes où MA50 est vide (NaN)
data = df[["MA50", "Volume"]].dropna()

# np.corrcoef : matrice de corrélation 2x2 (la diagonale vaut toujours 1)
corr_matrix = np.corrcoef(data["MA50"], data["Volume"])
print("Correlation matrix:\n", corr_matrix)

# [0, 1] : case ligne 0 / colonne 1 = le coefficient entre les 2 séries
print("Correlation MA50 vs Volume:", corr_matrix[0, 1])

In [ ]:
# === MATRICE DE CORRÉLATION COMPLÈTE (Pandas) ===
# .corr() : corrélation entre TOUTES les paires de colonnes d'un coup
df[["Close", "MA50", "MA200", "Volume", "Daily_Return"]].corr()

In [ ]:
# ============================================================
# 6. SYNTHÈSE ET ENSEIGNEMENTS
# ============================================================
#
# JEU DE DONNÉES
# - Cours quotidien de l'action Apple (AAPL) du 1981-01-02 au 2023-01-27.
# - 10 608 jours de bourse ; colonnes : Open, High, Low, Close, Adj Close, Volume.
# - Aucune valeur manquante.
#
# TRAJECTOIRE DU PRIX
# - Le prix de clôture reste quasi plat (bien sous 1 $, ajusté des splits) pendant
#   ~25 ans, puis croît de façon exponentielle à partir de ~2005 (ère iPod/iPhone),
#   jusqu'à un record de 182,01 $.
#
# STATISTIQUES DESCRIPTIVES (Close)
# - Moyenne = 16,70 $ mais Médiane = 0,49 $ -> écart énorme.
# - Forte asymétrie à droite : plus de la moitié des jours sont antérieurs à ~2005
#   à très bas prix, la moyenne étant tirée vers le haut par les prix récents.
#   Écart-type = 35,47 $.
#
# MOYENNES MOBILES
# - Les MA 50 et 200 jours confirment la tendance haussière de long terme.
# - Elles corrèlent à ~0,99 avec le prix (normal : une MA est un prix lissé).
#
# TEST D'HYPOTHÈSE (t-test : moyenne Close 2020 vs 2021)
# - Moyenne 2020 = 95,35 $, Moyenne 2021 = 140,99 $.
# - t = -27,59 ; p-value = 1,0e-102 (bien sous 0,05).
# - Conclusion : différence hautement significative ; l'action a coté bien plus
#   haut en 2021 qu'en 2020 (on rejette H0).
#
# RENDEMENTS QUOTIDIENS ET NORMALITÉ
# - Rendement quotidien moyen = +0,105 % ; volatilité quotidienne (std) = 2,82 %.
# - Test de normalité D'Agostino-Pearson : p-value ~ 0 -> rendements NON normaux.
# - Excès de kurtosis = 18,1 (normal = 0) : "queues épaisses", les mouvements
#   extrêmes sont bien plus fréquents que ne le prédit une loi normale.
# - Légère asymétrie négative (-0,375) : les chutes brutales sont un peu plus marquées.
#
# CORRÉLATIONS
# - Prix vs moyennes mobiles ~ 0,99 (trivial : les MA sont le prix lissé).
# - Volume vs prix ~ -0,22 : faible et négatif, biaisé par les splits et la tendance
#   de fond, donc corrélation != causalité.
# - Rendements quotidiens vs tout ~ 0 : quasi indépendants du niveau de prix ou du
#   volume, cohérent avec une marche aléatoire.

In [ ]:
# ============================================================
# 7. RÉFLEXION
# ============================================================
#
# DIFFICULTÉS RENCONTRÉES ET SOLUTIONS
#
# 1. Lecture des dates
#    - Format JJ/MM/AAAA (européen), mais pandas lit par défaut à l'américaine
#      (MM/JJ) et se trompait.
#    - Solution : pd.read_csv(..., parse_dates=["Date"], dayfirst=True), puis mettre
#      Date en index et trier chronologiquement.
#
# 2. Bibliothèques manquantes
#    - SciPy (tests) et mplfinance (chandeliers) n'étaient pas installés dans le
#      Python du notebook.
#    - Solution : installation depuis le notebook avec
#      !{sys.executable} -m pip install ..., en visant le bon noyau (kernel).
#
# 3. Chandeliers illisibles
#    - Un graphique en chandeliers sur 42 ans (10 608 bougies) est illisible.
#    - Solution : zoom sur les 60 derniers jours (df.tail(60)).
#
# 4. Choisir le bon test de normalité
#    - Avec 10 608 rendements, Shapiro-Wilk n'est pas fiable (prévu pour n <= 5000).
#    - Solution : D'Agostino-Pearson (scipy.stats.normaltest), adapté aux grands
#      échantillons.
#
# 5. Interpréter les corrélations avec prudence
#    - La corrélation négative prix-volume est faible et biaisée par les splits et la
#      tendance, donc pas lue comme une causalité.
#
# 6. Hypothèses du t-test
#    - Les prix quotidiens d'une année sont autocorrélés, donc pas strictement
#      indépendants. La conclusion reste claire vu l'écart, mais cette limite est à noter.
#
# À RETENIR
# - Combiner NumPy, Pandas, Matplotlib et SciPy permet de charger, visualiser et
#   tester statistiquement une longue série financière. Les "queues épaisses" des
#   rendements rappellent que les marchés réels ne suivent pas une loi normale.